# Homework 5.1 - Step 3: Hidden Glyph Bijection

This notebook constructs the artificial glyph cipher required for Task 3.

The source syllable vocabulary from Step 2 is

$$
T = \{t_1,\ldots,t_K\}.
$$

We construct an artificial glyph set

$$
G = \{g_1,\ldots,g_K\}
$$

and a seeded random bijection

$$
\Phi:T\rightarrow G.
$$

The same map is used for every occurrence of every source syllable.

The notebook then converts the entire *Meditations* syllable sequence into glyphs while copying all visible separators exactly as they appear in the cleaned corpus.

## Information separation

This step creates two different kinds of files.

### Decipherment inputs

These files may be used in later fitting:

- the glyph-only corpus,
- the glyph-token sequence,
- the glyph vocabulary.

They contain no plaintext syllable labels and no hidden glyph-to-syllable translations.

### Evaluation-only data

The complete hidden map is saved in a separate `evaluation_only/` directory.

It is needed to construct the artificial dataset and later evaluate the final decipherment, but it must not be loaded by the fitting procedure except when a later step explicitly creates the permitted anchor pairs.

The short aligned example printed near the end of this notebook is only a dataset-construction check. It is not saved as a decipherment-input file.

## Separator convention

Spaces, newlines, apostrophes, hyphens, commas, periods, quotation marks, and other punctuation are visible side information.

They are not syllable tokens and are not assigned glyph IDs.

If a normalized source fragment is conceptually

$$
x_1x_2,\ x_3
$$

then only the syllables $$x_1,x_2,x_3$$ are replaced by glyphs. The comma and space are copied unchanged.

This notebook uses exactly the same alphabetic-word convention and syllabification rule as Step 2.

In [1]:
from pathlib import Path
from importlib.metadata import version
import csv
import re

import numpy as np

try:
    import pyphen
except ImportError as exc:
    raise ImportError(
        "Pyphen is required because Step 3 must reproduce the exact Step 2 "
        "syllabification. Run `%pip install pyphen` if it is not installed."
    ) from exc

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "codes" else cwd
DATA_DIR = PROJECT_ROOT / "data"

MEDITATIONS_PATH = DATA_DIR / "meditations_tokenization_text.txt"
SOURCE_VOCAB_PATH = DATA_DIR / "source_syllable_vocabulary.csv"
SOURCE_SEQUENCE_PATH = DATA_DIR / "meditations_syllable_token_sequence.csv"

DECIPHER_DIR = DATA_DIR / "decipherment_inputs"
EVALUATION_DIR = DATA_DIR / "evaluation_only"

DECIPHER_DIR.mkdir(parents=True, exist_ok=True)
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

required_files = [
    MEDITATIONS_PATH,
    SOURCE_VOCAB_PATH,
    SOURCE_SEQUENCE_PATH,
]

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required file: {path}\n"
            "Run Step 1 and Step 2 successfully before Step 3."
        )

print("Pyphen version:", version("pyphen"))
print("Decipherment-input folder:", DECIPHER_DIR.resolve())
print("Evaluation-only folder:", EVALUATION_DIR.resolve())

Pyphen version: 0.18.1
Decipherment-input folder: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\decipherment_inputs
Evaluation-only folder: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\evaluation_only


## Load and verify the Step 2 source vocabulary

In [2]:
source_rows = []

with SOURCE_VOCAB_PATH.open("r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)

    for row in reader:
        source_rows.append({
            "source_id": int(row["source_id"]),
            "syllable": row["syllable"],
        })

source_rows.sort(key=lambda row: row["source_id"])

source_ids = [row["source_id"] for row in source_rows]
source_syllables = [row["syllable"] for row in source_rows]

K = len(source_rows)

assert source_ids == list(range(K)), (
    "Source IDs are not exactly 0,1,...,K-1."
)

assert len(set(source_syllables)) == K, (
    "The Step 2 source vocabulary contains duplicate syllables."
)

source_id_by_syllable = {
    row["syllable"]: row["source_id"]
    for row in source_rows
}

print("K =", f"{K:,}", "distinct source syllables")
print("Verified one unique source ID per syllable.")

K = 3,052 distinct source syllables
Verified one unique source ID per syllable.


## Load and verify the Step 2 source syllable sequence

In [3]:
step2_syllable_sequence = []
step2_source_ids = []

with SOURCE_SEQUENCE_PATH.open("r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)

    for expected_position, row in enumerate(reader):
        position = int(row["position"])
        source_id = int(row["source_id"])
        syllable = row["syllable"]

        assert position == expected_position
        assert source_id_by_syllable[syllable] == source_id

        step2_syllable_sequence.append(syllable)
        step2_source_ids.append(source_id)

L = len(step2_syllable_sequence)

print("L =", f"{L:,}", "source syllable occurrences")
print("Verified the Step 2 token sequence and source IDs.")

L = 77,260 source syllable occurrences
Verified the Step 2 token sequence and source IDs.


## Construct the hidden bijection

Create one abstract glyph for each source syllable.

The glyph labels are neutral labels such as `G0001`; they contain no syllable information.

A seeded random permutation assigns the glyphs to source syllables. The fixed random seed makes the artificial cipher reproducible.

In [4]:
RANDOM_SEED = 63803

rng = np.random.default_rng(RANDOM_SEED)

permuted_glyph_ids = rng.permutation(K)

glyph_width = max(4, len(str(K - 1)))


def glyph_label(glyph_id):
    return f"G{glyph_id:0{glyph_width}d}"


phi = {}
inverse_phi = {}

for row, glyph_id in zip(source_rows, permuted_glyph_ids):
    syllable = row["syllable"]
    glyph_id = int(glyph_id)
    label = glyph_label(glyph_id)

    phi[syllable] = {
        "glyph_id": glyph_id,
        "glyph_label": label,
    }

    inverse_phi[glyph_id] = {
        "syllable": syllable,
        "source_id": row["source_id"],
    }

assert len(phi) == K
assert len(inverse_phi) == K
assert len({item["glyph_id"] for item in phi.values()}) == K

print("Random seed:", RANDOM_SEED)
print("Constructed a bijection with", f"{K:,}", "syllables and", f"{K:,}", "glyphs.")
print("Bijection checks passed.")

Random seed: 63803
Constructed a bijection with 3,052 syllables and 3,052 glyphs.
Bijection checks passed.


## Save the complete map as evaluation-only ground truth

This file contains the complete hidden answer key.

Do not load this file in the later fitting procedure.

Later, the anchor-generation step may use this ground-truth file to create only the explicitly revealed anchor pairs.

In [5]:
HIDDEN_MAP_PATH = EVALUATION_DIR / "hidden_syllable_glyph_bijection.csv"

with HIDDEN_MAP_PATH.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    writer.writerow([
        "source_id",
        "syllable",
        "glyph_id",
        "glyph_label",
    ])

    for row in source_rows:
        syllable = row["syllable"]
        mapping = phi[syllable]

        writer.writerow([
            row["source_id"],
            syllable,
            mapping["glyph_id"],
            mapping["glyph_label"],
        ])

print("Saved evaluation-only hidden map:")
print(HIDDEN_MAP_PATH.resolve())

Saved evaluation-only hidden map:
C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\evaluation_only\hidden_syllable_glyph_bijection.csv


## Save the visible glyph vocabulary without plaintext labels

In [6]:
GLYPH_VOCAB_PATH = DECIPHER_DIR / "glyph_vocabulary.csv"

with GLYPH_VOCAB_PATH.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    writer.writerow([
        "glyph_id",
        "glyph_label",
    ])

    for glyph_id in range(K):
        writer.writerow([
            glyph_id,
            glyph_label(glyph_id),
        ])

print("Saved glyph-only vocabulary:")
print(GLYPH_VOCAB_PATH.resolve())

Saved glyph-only vocabulary:
C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\decipherment_inputs\glyph_vocabulary.csv


## Reproduce the Step 2 syllabification rule

In [7]:
HYPHENATOR = pyphen.Pyphen(lang="en_US")
VOWELS = set("aeiouy")


def fallback_written_syllables(word):
    if not word:
        return []

    nuclei = []
    i = 0

    while i < len(word):
        if word[i] in VOWELS:
            start = i

            while i + 1 < len(word) and word[i + 1] in VOWELS:
                i += 1

            end = i + 1
            nuclei.append((start, end))

        i += 1

    if len(nuclei) <= 1:
        return [word]

    boundaries = []

    for j in range(len(nuclei) - 1):
        current_end = nuclei[j][1]
        next_start = nuclei[j + 1][0]
        cluster_length = next_start - current_end

        if cluster_length <= 1:
            boundary = current_end
        else:
            boundary = current_end + 1

        boundaries.append(boundary)

    pieces = []
    start = 0

    for boundary in boundaries:
        pieces.append(word[start:boundary])
        start = boundary

    pieces.append(word[start:])
    pieces = [piece for piece in pieces if piece]

    if "".join(pieces) != word:
        raise RuntimeError(
            f"Fallback reconstruction failed for {word!r}"
        )

    return pieces


def syllabify_word(word):
    try:
        inserted = HYPHENATOR.inserted(word)
        segments = [
            part for part in inserted.split("-")
            if part
        ]

        if segments and "".join(segments) == word:
            return segments

    except Exception:
        pass

    segments = fallback_written_syllables(word)

    if "".join(segments) != word:
        raise RuntimeError(
            f"Reconstruction failed for {word!r}"
        )

    return segments

## Convert the full source text while preserving separators

Step 3 reads the frozen `meditations_tokenization_text.txt` produced by Step 2, so the exact same ligature normalization is used in both tasks.

The regular expression separates the normalized source into:

1. ASCII English word spans `[a-z]+`,
2. all other visible spans.

Only the English word spans are syllabified and replaced by glyphs.

Whitespace and punctuation are copied character-for-character. Any remaining non-ASCII alphabetic material is also left untouched and is treated as a reported nonsyllabic source artifact rather than silently converted into a syllable token.

In [8]:
source_text = MEDITATIONS_PATH.read_text(encoding="utf-8")

SPAN_RE = re.compile(r"[a-z]+|[^a-z]+")


def encode_source_text(text):
    output_parts = []
    encoded_syllables = []
    encoded_glyph_ids = []
    encoded_glyph_labels = []
    copied_separator_spans = []
    word_records = []

    for match in SPAN_RE.finditer(text):
        span = match.group(0)

        if re.fullmatch(r"[a-z]+", span):
            word = span
            syllables = syllabify_word(word)

            assert "".join(syllables) == word

            labels = []

            for syllable in syllables:
                if syllable not in phi:
                    raise KeyError(
                        f"Syllable {syllable!r} was not found in "
                        "the fixed Step 2 source vocabulary."
                    )

                glyph_id = phi[syllable]["glyph_id"]
                label = phi[syllable]["glyph_label"]

                encoded_syllables.append(syllable)
                encoded_glyph_ids.append(glyph_id)
                encoded_glyph_labels.append(label)
                labels.append(label)

            output_parts.append("".join(labels))

            word_records.append({
                "word": word,
                "syllables": syllables,
                "glyph_labels": labels,
            })

        else:
            output_parts.append(span)
            copied_separator_spans.append(span)

    glyph_text = "".join(output_parts)

    return {
        "glyph_text": glyph_text,
        "syllables": encoded_syllables,
        "glyph_ids": encoded_glyph_ids,
        "glyph_labels": encoded_glyph_labels,
        "separator_spans": copied_separator_spans,
        "word_records": word_records,
    }


encoded = encode_source_text(source_text)

print("Encoded full Meditations corpus.")
print("Glyph-token occurrences:", f"{len(encoded['glyph_ids']):,}")
print("Copied separator spans:", f"{len(encoded['separator_spans']):,}")

Encoded full Meditations corpus.
Glyph-token occurrences: 77,260
Copied separator spans: 57,029


## Verify exact agreement with Step 2

In [9]:
assert encoded["syllables"] == step2_syllable_sequence, (
    "The Step 3 syllable sequence does not exactly match Step 2."
)

assert len(encoded["glyph_ids"]) == L

for syllable, glyph_id in zip(
    encoded["syllables"],
    encoded["glyph_ids"],
):
    assert phi[syllable]["glyph_id"] == glyph_id

print("Step 3 reproduces the exact Step 2 syllable sequence.")
print("Every one of the L syllable occurrences received exactly one glyph.")

Step 3 reproduces the exact Step 2 syllable sequence.
Every one of the L syllable occurrences received exactly one glyph.


## Verify that visible separators were copied unchanged

In [10]:
original_separator_spans = [
    match.group(0)
    for match in SPAN_RE.finditer(source_text)
    if re.fullmatch(r"[a-z]+", match.group(0)) is None
]

assert encoded["separator_spans"] == original_separator_spans

original_separator_text = "".join(original_separator_spans)
copied_separator_text = "".join(encoded["separator_spans"])

assert copied_separator_text == original_separator_text

print("Separator verification passed.")
print("All punctuation and whitespace spans were copied unchanged.")

Separator verification passed.
All punctuation and whitespace spans were copied unchanged.


## Save the glyph-only corpus

In [11]:
GLYPH_CORPUS_PATH = DECIPHER_DIR / "meditations_glyph_corpus.txt"

GLYPH_CORPUS_PATH.write_text(
    encoded["glyph_text"],
    encoding="utf-8",
)

print("Saved glyph-only corpus:")
print(GLYPH_CORPUS_PATH.resolve())

Saved glyph-only corpus:
C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\decipherment_inputs\meditations_glyph_corpus.txt


## Save the glyph-token sequence

In [12]:
GLYPH_SEQUENCE_PATH = DECIPHER_DIR / "meditations_glyph_token_sequence.csv"

with GLYPH_SEQUENCE_PATH.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    writer.writerow([
        "position",
        "glyph_id",
        "glyph_label",
    ])

    for position, (glyph_id, label) in enumerate(
        zip(
            encoded["glyph_ids"],
            encoded["glyph_labels"],
        )
    ):
        writer.writerow([
            position,
            glyph_id,
            label,
        ])

print("Saved glyph-token sequence:")
print(GLYPH_SEQUENCE_PATH.resolve())

Saved glyph-token sequence:
C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\decipherment_inputs\meditations_glyph_token_sequence.csv


## Short aligned dataset-construction check

The following output is deliberately small and is displayed only to verify that the artificial dataset was constructed correctly.

It shows:

- the normalized source word,
- its written syllable segmentation,
- the corresponding glyph labels.

This aligned information is not written into `decipherment_inputs/` and must not be supplied to the fitting procedure.

In [13]:
EXAMPLE_WORD_COUNT = 10

print(
    f"{'Word':<22} "
    f"{'Syllables':<38} "
    "Glyph encoding"
)
print("-" * 110)

for record in encoded["word_records"][:EXAMPLE_WORD_COUNT]:
    syllable_text = " | ".join(record["syllables"])
    glyph_text = " ".join(record["glyph_labels"])

    print(
        f"{record['word']:<22} "
        f"{syllable_text:<38} "
        f"{glyph_text}"
    )

print()
print(
    "Dataset-construction check only. "
    "Do not use this alignment as decipherment input."
)

Word                   Syllables                              Glyph encoding
--------------------------------------------------------------------------------------------------------------
of                     of                                     G2196
my                     my                                     G0091
grandfather            grand | fa | ther                      G0841 G0741 G1322
verus                  verus                                  G2727
i                      i                                      G2939
have                   have                                   G1611
learned                learned                                G1993
to                     to                                     G1887
be                     be                                     G2004
gentle                 gen | tle                              G0688 G2877

Dataset-construction check only. Do not use this alignment as decipherment input.


## Check that glyph labels themselves reveal no syllable names

In [14]:
label_pattern = re.compile(
    rf"^G\d{{{glyph_width}}}$"
)

assert all(
    label_pattern.fullmatch(glyph_label(glyph_id))
    for glyph_id in range(K)
)

print(
    "All glyph names use only a neutral G + numeric identifier format."
)
print(
    "The syllable-to-glyph assignment is determined only by the seeded random permutation."
)

All glyph names use only a neutral G + numeric identifier format.
The syllable-to-glyph assignment is determined only by the seeded random permutation.


## Save reproducibility metadata

In [15]:
METADATA_PATH = DATA_DIR / "task3_glyph_metadata.csv"

metadata_rows = [
    ["random_seed", RANDOM_SEED],
    ["source_vocabulary_size_K", K],
    ["source_syllable_count_L", L],
    ["glyph_vocabulary_size", K],
    ["glyph_label_width", glyph_width],
    ["bijection_verified", True],
    ["step2_sequence_match_verified", True],
    ["separators_preserved_verified", True],
    ["hidden_map_location", "evaluation_only/hidden_syllable_glyph_bijection.csv"],
    ["glyph_corpus_location", "decipherment_inputs/meditations_glyph_corpus.txt"],
    ["glyph_sequence_location", "decipherment_inputs/meditations_glyph_token_sequence.csv"],
]

with METADATA_PATH.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["quantity", "value"])
    writer.writerows(metadata_rows)

print("Saved:")
print(METADATA_PATH.resolve())

Saved:
C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\task3_glyph_metadata.csv


## Task 3 outputs

After this notebook runs successfully, the relevant files are:

### Allowed decipherment inputs

`data/decipherment_inputs/`

- `glyph_vocabulary.csv`
- `meditations_glyph_corpus.txt`
- `meditations_glyph_token_sequence.csv`

These files do not contain plaintext syllable labels.

### Evaluation-only ground truth

`data/evaluation_only/`

- `hidden_syllable_glyph_bijection.csv`

This file contains the complete hidden map and must not be supplied to the fitting procedure.

### Metadata

`data/task3_glyph_metadata.csv`

The short aligned example appears only in the notebook output as a construction check.

Step 4 should split the glyph corpus into training, validation, and test portions without using the evaluation-only plaintext map.